In [ ]:
from pathlib import Path
import zipfile
import shutil
import json

import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

DATA_DIR = BASE_DIR / "data"
COLAB_DIR = DATA_DIR / "imgsMIIA_colab"

RAW_ZIPS_DIR = COLAB_DIR / "raw_zips"
EXTRACTED_DIR = COLAB_DIR / "extracted"
PREPARED_DIR = COLAB_DIR / "prepared"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

for carpeta in [RAW_ZIPS_DIR, EXTRACTED_DIR, PREPARED_DIR, TABLES_DIR, FIGURES_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

print("RAW_ZIPS_DIR:", RAW_ZIPS_DIR)
print("EXTRACTED_DIR:", EXTRACTED_DIR)
print("RAW_ZIPS_DIR existe:", RAW_ZIPS_DIR.exists())

In [ ]:
zip_files = list(RAW_ZIPS_DIR.glob("*.zip"))

print("Número de zips encontrados:", len(zip_files))

for z in zip_files:
    print(z.name)

In [ ]:
resumen_zips = []

for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        nombres = z.namelist()
        
        n_json = sum(1 for n in nombres if n.lower().endswith(".json"))
        n_jpg = sum(1 for n in nombres if n.lower().endswith((".jpg", ".jpeg", ".png")))
        n_html = sum(1 for n in nombres if n.lower().endswith(".html"))
        
        resumen_zips.append({
            "zip": zip_path.name,
            "total_archivos": len(nombres),
            "imagenes": n_jpg,
            "json": n_json,
            "html": n_html
        })

df_zips = pd.DataFrame(resumen_zips)
display(df_zips)

csv_path = TABLES_DIR / "imgsMIIA_resumen_zips.csv"
df_zips.to_csv(csv_path, index=False)

print("Resumen guardado en:", csv_path)

In [ ]:
for zip_path in zip_files:
    nombre_carpeta = zip_path.stem
    destino = EXTRACTED_DIR / nombre_carpeta
    
    if destino.exists():
        shutil.rmtree(destino)
    
    destino.mkdir(parents=True, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(destino)

print("Zips extraídos en:", EXTRACTED_DIR)

In [ ]:
ext_img = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]

filas = []

for carpeta_zip in EXTRACTED_DIR.iterdir():
    if not carpeta_zip.is_dir():
        continue
    
    imagenes = []
    jsons = []
    
    for p in carpeta_zip.rglob("*"):
        if p.suffix in ext_img:
            imagenes.append(p)
        elif p.suffix.lower() == ".json":
            jsons.append(p)
    
    filas.append({
        "carpeta": carpeta_zip.name,
        "imagenes": len(imagenes),
        "json": len(jsons)
    })

df_extraidos = pd.DataFrame(filas)
display(df_extraidos)

csv_path = TABLES_DIR / "imgsMIIA_resumen_extraidos.csv"
df_extraidos.to_csv(csv_path, index=False)

print("Resumen guardado en:", csv_path)

In [ ]:
json_colab = list(EXTRACTED_DIR.rglob("*.json"))

print("JSON encontrados:", len(json_colab))

for jp in json_colab[:50]:
    print(jp.relative_to(EXTRACTED_DIR))

In [ ]:
def detectar_formato_json(json_path):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except:
        return "error_lectura"
    
    if isinstance(data, dict):
        claves = list(data.keys())
        
        if "_via_img_metadata" in claves:
            return "via_project"
        
        if "imagePath" in claves and "shapes" in claves:
            return "labelme"
        
        # Algunos exports de VIA son diccionarios con filename/regions dentro
        for k in claves[:5]:
            item = data[k]
            if isinstance(item, dict) and "filename" in item and "regions" in item:
                return "via_annotations"
        
        return "dict_otro"
    
    if isinstance(data, list):
        return "lista"
    
    return "otro"


filas = []

for jp in json_colab:
    formato = detectar_formato_json(jp)
    
    filas.append({
        "json_path": str(jp),
        "json_relativo": str(jp.relative_to(EXTRACTED_DIR)),
        "formato": formato
    })

df_json = pd.DataFrame(filas)

display(df_json["formato"].value_counts())
display(df_json.head(20))

csv_path = TABLES_DIR / "imgsMIIA_formatos_json.csv"
df_json.to_csv(csv_path, index=False)

print("CSV guardado en:", csv_path)

In [ ]:
def detectar_clase_por_texto(texto):
    texto = texto.upper()
    
    if "_DT" in texto or "- DT" in texto or " DT" in texto or "DORSAL" in texto:
        return "DT"
    
    if "_VT" in texto or "- VT" in texto or " VT" in texto or "VENTRAL" in texto:
        return "VT"
    
    if "_LT" in texto or "- LT" in texto or " LT" in texto or "LATERAL" in texto:
        return "LT"
    
    return "desconocida"


df_json["clase_detectada"] = df_json["json_relativo"].apply(detectar_clase_por_texto)

display(df_json["clase_detectada"].value_counts())
display(df_json[["json_relativo", "formato", "clase_detectada"]].head(30))

In [ ]:
imagenes_colab = []

for p in EXTRACTED_DIR.rglob("*"):
    if p.suffix in ext_img:
        clase = detectar_clase_por_texto(str(p))
        
        imagenes_colab.append({
            "path": p,
            "relativa": str(p.relative_to(EXTRACTED_DIR)),
            "clase_detectada": clase
        })

df_imgs_colab = pd.DataFrame(imagenes_colab)

print("Imágenes encontradas:", len(df_imgs_colab))
display(df_imgs_colab["clase_detectada"].value_counts())
display(df_imgs_colab.head())

In [ ]:
muestras = df_imgs_colab.sample(
    n=min(8, len(df_imgs_colab)),
    random_state=7
)

plt.figure(figsize=(12, 8))

for i, (_, fila) in enumerate(muestras.iterrows()):
    img = cv2.imread(str(fila["path"]), cv2.IMREAD_COLOR)
    
    if img is None:
        continue
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(2, 4, i + 1)
    plt.imshow(img_rgb)
    plt.title(fila["clase_detectada"])
    plt.axis("off")

plt.tight_layout()

fig_path = FIGURES_DIR / "imgsMIIA_colab_muestras.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)